In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'

# Template registration project (must be allowed in engine settings)
template_project_name = None

# Engine name for template registration
engine_name = None

# Workflow ZIP path
workflow_zip_path = 'resources/simple-approval-workflow.zip'

# Template registrar credentials
idp_name_registrar = 'FakeCAS'
idp_username_registrar = None
idp_password_registrar = None

# Visibility setting: 'institution' or 'public'
visibility = 'institution'

# Test user credentials (the user who will test visibility)
idp_name_test = 'FakeCAS'
idp_username_test = None
idp_password_test = None

# Expected result: whether the template should be visible to test user
expect_visible = True

default_result_path = None
close_on_fail = False
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False

In [ ]:
import tempfile

if idp_username_registrar is None:
    idp_username_registrar = input(prompt=f'Username for registrar ({idp_name_registrar})')
if idp_password_registrar is None:
    idp_password_registrar = getpass(prompt=f'Password for {idp_username_registrar}@{idp_name_registrar}')

if idp_username_test is None:
    idp_username_test = input(prompt=f'Username for test user ({idp_name_test})')
if idp_password_test is None:
    idp_password_test = getpass(prompt=f'Password for {idp_username_test}@{idp_name_test}')

if template_project_name is None:
    template_project_name = input(prompt='Template project name')
if engine_name is None:
    engine_name = input(prompt='Workflow engine name')

workflow_name = datetime.now().strftime(f'VISIBILITY-TEST-%Y%m%d%H%M')
test_project_name = datetime.now().strftime(f'TEST-VISIBILITY-%Y%m%d%H%M')

# ワークフローテンプレートの公開範囲

- サブシステム名: アドオン
- ページ/アドオン: Workflow
- 機能分類: 公開範囲
- シナリオ名: 公開範囲に応じたテンプレートの可視性の検証
- 用意するテストデータ: アカウント（テンプレート登録者、テストユーザー）、テンプレート登録用プロジェクト
- 事前条件: エンジン登録が完了しており、テンプレート登録用プロジェクトがアップロード許可ノードIDに登録されていること

In [ ]:
work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import asyncio
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## 新規タブを開き、GRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    button = page.locator('//button[text() = "同意する"]')
    await button.scroll_into_view_if_needed()
    await button.click()
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step, new_page=True)

## テンプレート登録者としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_registrar, idp_username_registrar, idp_password_registrar, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧からテンプレート登録用プロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
template_project_url = None

async def _step(page):
    project_item = page.locator(f'//*[@data-test-dashboard-item-title and text() = "{template_project_name}"]')
    await project_item.scroll_into_view_if_needed()
    await project_item.click()
    title = page.locator('//span[@id = "nodeTitleEditable"]')
    await expect(title).to_be_visible(timeout=transition_timeout)
    await title.scroll_into_view_if_needed()
    global template_project_url
    template_project_url = page.url

await run_pw(_step)

## 「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    addons_link = page.locator('//a[text() = "アドオン"]')
    await addons_link.scroll_into_view_if_needed()
    await addons_link.click()
    header = page.locator('//h3[text() = "アドオンを選択"]')
    await expect(header).to_be_visible(timeout=transition_timeout)
    await header.scroll_into_view_if_needed()

await run_pw(_step)

## 「Workflow」の「有効にする」をクリックする

Workflowアドオンが有効化されること（すでに有効な場合はスキップ）

In [ ]:
async def _step(page):
    enable_button = page.locator('//div[@full_name = "Workflow"]//a[text() = "有効にする"]')
    if await enable_button.count():
        await enable_button.scroll_into_view_if_needed()
        await enable_button.click()
        confirm_button = page.locator('//button[@data-bb-handler = "confirm"]')
        await expect(confirm_button).to_be_visible(timeout=transition_timeout)
        await confirm_button.scroll_into_view_if_needed()
        await confirm_button.click()
    engine_select = page.locator('#workflow-engine-id')
    await expect(engine_select).to_be_visible(timeout=transition_timeout)
    await engine_select.scroll_into_view_if_needed()

await run_pw(_step)

## 「ワークフローエンジン」ドロップダウンからエンジンを選択する

指定したエンジンが選択されること

In [ ]:
async def _step(page):
    engine_select = page.locator('#workflow-engine-id')
    await engine_select.scroll_into_view_if_needed()
    option = page.locator(f'#workflow-engine-id option:has-text("{engine_name}")')
    value = await option.get_attribute('value')
    await engine_select.select_option(value=value)

await run_pw(_step)

## 「ワークフローZIP」にBPMNファイルを含むZIPをアップロードする

ZIPファイルが選択されること

In [ ]:
async def _step(page):
    zip_input = page.locator('#workflow-zip')
    await zip_input.scroll_into_view_if_needed()
    await zip_input.set_input_files(workflow_zip_path)

await run_pw(_step)

## 「名前」にワークフロー名を入力する

ワークフロー名が入力されること

In [ ]:
async def _step(page):
    name_input = page.locator('#workflow-label')
    await name_input.scroll_into_view_if_needed()
    await name_input.fill(workflow_name)

await run_pw(_step)

## 「公開範囲」ドロップダウンで指定された公開範囲を選択する

指定された公開範囲が選択されること

In [ ]:
async def _step(page):
    visibility_select = page.locator('#workflow-visibility')
    await visibility_select.scroll_into_view_if_needed()
    await visibility_select.select_option(value=visibility)

await run_pw(_step)

## 「管理者トークン」ドロップダウンで「閲覧＆編集権限で使用」を選択する

「閲覧＆編集権限で使用」が選択されること

In [ ]:
async def _step(page):
    token_mode_select = page.locator('select[data-bind="value: form.managerTokenMode"]')
    await token_mode_select.scroll_into_view_if_needed()
    await token_mode_select.select_option(value='readwrite')

await run_pw(_step)

## 「ワークフローテンプレートを登録」をクリックする

テンプレートが作成され、ローカルワークフロー一覧に表示されること

In [ ]:
async def _step(page):
    register_button = page.locator('//button[.//span[contains(text(), "ワークフローテンプレートを登録")]]')
    await register_button.scroll_into_view_if_needed()
    await register_button.click()
    template_row = page.locator(f'#localWorkflowsPanel').locator(f'//td//strong[text()="{workflow_name}"]')
    await expect(template_row).to_be_visible(timeout=transition_timeout)
    await template_row.scroll_into_view_if_needed()

await run_pw(_step)

## テンプレート登録者をログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_registrar, transition_timeout=transition_timeout)

await run_pw(_step)

## テストユーザーとしてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await grdm.login(page, idp_name_test, idp_username_test, idp_password_test, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## 新規プロジェクトを作成する

テスト用プロジェクトが作成されること

In [ ]:
async def _step(page):
    await grdm.ensure_project_exists(page, test_project_name, transition_timeout=transition_timeout)
    project_item = page.locator(f'//*[@data-test-dashboard-item-title and text() = "{test_project_name}"]')
    await project_item.scroll_into_view_if_needed()
    await project_item.click()
    title = page.locator('//span[@id = "nodeTitleEditable"]')
    await expect(title).to_be_visible(timeout=transition_timeout)
    await title.scroll_into_view_if_needed()

await run_pw(_step)

## 「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    addons_link = page.locator('//a[text() = "アドオン"]')
    await addons_link.scroll_into_view_if_needed()
    await addons_link.click()
    header = page.locator('//h3[text() = "アドオンを選択"]')
    await expect(header).to_be_visible(timeout=transition_timeout)
    await header.scroll_into_view_if_needed()

await run_pw(_step)

## 「Workflow」の「有効にする」をクリックする

Workflowアドオンが有効化されること

In [ ]:
async def _step(page):
    enable_button = page.locator('//div[@full_name = "Workflow"]//a[text() = "有効にする"]')
    if await enable_button.count():
        await enable_button.scroll_into_view_if_needed()
        await enable_button.click()
        confirm_button = page.locator('//button[@data-bb-handler = "confirm"]')
        await expect(confirm_button).to_be_visible(timeout=transition_timeout)
        await confirm_button.scroll_into_view_if_needed()
        await confirm_button.click()
    activations_tab = page.locator('//*[@data-target="#activationsPanel"]')
    await expect(activations_tab).to_be_visible(timeout=transition_timeout)
    await activations_tab.scroll_into_view_if_needed()

await run_pw(_step)

## テンプレートの可視性を検証する

expect_visible=Trueの場合: テンプレートが有効化用ドロップダウンに表示されること
expect_visible=Falseの場合: テンプレートが有効化用ドロップダウンに表示されないこと

In [ ]:
async def _step(page):
    select_element = page.locator('#activate-workflow-select')
    await select_element.scroll_into_view_if_needed()
    template_option = page.locator(f'#activate-workflow-select option:has-text("{workflow_name}")')
    if expect_visible:
        await expect(template_option).to_be_attached(timeout=transition_timeout)
    else:
        # Ensure dropdown is loaded by checking the select element exists
        await expect(select_element).to_be_visible(timeout=transition_timeout)
        await expect(template_option).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## テストユーザーをログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_registrar, transition_timeout=transition_timeout)

await run_pw(_step)

## テンプレート管理プロジェクトのURLを開き、テンプレート登録者でログインする

In [ ]:
async def _step(page):
    await page.goto(template_project_url)
    await grdm.login(page, idp_name_registrar, idp_username_registrar, idp_password_registrar, transition_timeout=transition_timeout)

await run_pw(_step)

## 「アドオン」をクリックし、Workflowアドオンのテンプレート一覧で対象行の無効化ボタンをクリックし、ダイアログで確認ボタンをクリックする
ステータスが「無効」に変わること

In [ ]:
async def _step(page):
    addons_link = page.locator('//a[text() = "アドオン"]')
    await addons_link.scroll_into_view_if_needed()
    await addons_link.click()
    header = page.locator('//h3[text() = "アドオンを選択"]')
    await expect(header).to_be_visible(timeout=transition_timeout)
    row = page.locator('#localWorkflowsPanel').locator(f'//tr[.//strong[text()="{workflow_name}"]]')
    await row.locator('//button[*[text() = "無効にする"]]').click()
    await expect(page.locator('//h4[text() = "ワークフローテンプレートの無効化"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[contains(@data-bind, "confirmDisableTemplate")]').click()
    await expect(row.locator('//*[contains(@class, "label") and text() = "無効"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)

## テンプレート一覧で対象行の削除ボタンをクリックし、確認ダイアログで「削除」をクリックする

テンプレート一覧から削除されること

In [ ]:
async def _step(page):
    row = page.locator('#localWorkflowsPanel').locator(f'//tr[.//strong[text()="{workflow_name}"]]')
    await row.locator('//button[contains(@data-bind, "deleteTemplate")]').click()
    await expect(page.locator('//h4[text() = "ワークフローテンプレートの削除"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[contains(@data-bind, "confirmDeleteTemplate")]').click()
    await expect(page.locator(f'#localWorkflowsPanel').locator(f'//td//strong[text()="{workflow_name}"]')).to_have_count(0, timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)

## 後始末

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}